## **Schema Inspection — All 8 Bronze Files**

In [ ]:
bronze_path = "abfss://246f1c03-56af-4f6f-8640-aff9681c241b@onelake.dfs.fabric.microsoft.com/656f0bb9-d036-4f99-9e67-a4d0155df259/Files/Raw_Data_Bronze"

tables = ["currencyexchange", "customer", "date", "orderrows", 
          "orders", "product", "sales", "store"]

for table in tables:
    print(f"\n{'='*50}")
    print(f"TABLE: {table}")
    print('='*50)
    df = spark.read.format("delta").load(f"{bronze_path}/{table}")
    df.printSchema()
    print(f"Row count: {df.count()}")
    print('='*50)
    print("Succesfully Inspected")

## **Registered the Table inside the Views**

In [ ]:
bronze_path = "abfss://246f1c03-56af-4f6f-8640-aff9681c241b@onelake.dfs.fabric.microsoft.com/656f0bb9-d036-4f99-9e67-a4d0155df259/Files/Raw_Data_Bronze"

tables = ["currencyexchange", "customer", "date", "orderrows", 
          "orders", "product", "sales", "store"]

for table in tables:
    df = spark.read.format("delta").load(f"{bronze_path}/{table}")
    df.createOrReplaceTempView(f"bronze_{table}")
    print(f"✅ Registered: bronze_{table}")

## **Explore the Currency Exchange table**

In [ ]:
# Explore the table content
display(spark.sql("SELECT * FROM bronze_currencyexchange LIMIT 10"))

# Row count
spark.sql("SELECT COUNT(*) AS TotalRows FROM bronze_currencyexchange").show()

## **Silver: currencyexchange - Transformation**

In [ ]:
silver_currencyexchange = spark.sql("""
    SELECT
        CAST(Date AS DATE) AS Date,
        FromCurrency,
        ToCurrency,
        CAST(Exchange AS DOUBLE) AS ExchangeRate
    FROM bronze_currencyexchange
    WHERE Date IS NOT NULL
      AND Exchange IS NOT NULL
""")

silver_currencyexchange.write.format("delta").mode("overwrite").saveAsTable("Silver_Currencyexchange")
print(f"✅ silver_currencyexchange written — {silver_currencyexchange.count()} rows")